# Statistical uncertainty: Full Proposed 3D CNN vs No-Sampler ablation

This notebook performs a paired fixed-test ImageID-level analysis using only the saved probabilities from five finalized checkpoints per variant. It performs no training, model loading, or inference. The difference direction is always **Full Proposed - No-Sampler**.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import ast, hashlib, json, platform, re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path("/media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM")
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "Analysis" / "Statistical_Uncertainty_Proposed_vs_NoSampler.ipynb"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "Analysis" / "Statistical_Uncertainty_Proposed_vs_NoSampler"
INPUTS_DIR = OUTPUT_ROOT / "inputs"
BOOTSTRAP_DIR = OUTPUT_ROOT / "bootstrap"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
AUDIT_DIR = OUTPUT_ROOT / "audit"
for directory in [INPUTS_DIR, BOOTSTRAP_DIR, TABLES_DIR, FIGURES_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROPOSED_ROOT = PROJECT_ROOT / "Results" / "Proposed_Model"
NO_SAMPLER_ROOT = PROJECT_ROOT / "Results" / "Proposed_Model_No_Sampler"
PROPOSED_NOTEBOOK = PROJECT_ROOT / "notebooks" / "stage1_v2.ipynb"
NO_SAMPLER_NOTEBOOK = PROJECT_ROOT / "notebooks" / "stage1_v2_no_sampler.ipynb"
TEST_SPLIT = PROJECT_ROOT / "folds" / "fold_1" / "test.csv"
CLASS_NAMES = ["AD", "CN", "EMCI", "LMCI", "MCI", "SMC"]
DISPLAY_ORDER = ["CN", "SMC", "EMCI", "MCI", "LMCI", "AD"]
LABEL_MAP = {name: index for index, name in enumerate(CLASS_NAMES)}
FOLDS = [1, 2, 3, 4, 5]
N_IMAGES = 719
N_BOOTSTRAPS = 5000
BOOTSTRAP_SEED = 42
PLOT_DPI = 600
DIFFERENCE_DIRECTION = "Full Proposed - No-Sampler"
CORE_METRICS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "MCC", "Macro AUC"]
SMC_METRICS = ["SMC Precision", "SMC Recall", "SMC F1", "SMC AUC"]
CLASS_F1_METRICS = [f"{name} F1" for name in CLASS_NAMES]
ALL_METRICS = CORE_METRICS + SMC_METRICS + [key for key in CLASS_F1_METRICS if key != "SMC F1"]
PRIMARY_METRICS = ["Macro F1", "MCC", "Accuracy", "Macro AUC"]
PREDEFINED_METRICS = ["SMC F1", "SMC Recall"]
REPORT_METRICS = CORE_METRICS + SMC_METRICS

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False), encoding="utf-8")

CONFIG = {
    "analysis": "Paired bootstrap comparison of Full Proposed 3D CNN vs No-Sampler ablation",
    "difference_direction": DIFFERENCE_DIRECTION,
    "fixed_test_size": N_IMAGES,
    "checkpoints_per_variant": 5,
    "n_bootstraps": N_BOOTSTRAPS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "confidence_interval": "95% percentile bootstrap",
    "resampling_unit": "fixed-test MRI ImageID",
    "aggregation": "metric computed separately for each checkpoint, then averaged across five checkpoints",
    "ensemble_analysis": False,
    "training_performed": False,
    "inference_performed": False,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "class_mapping": LABEL_MAP,
    "manuscript_display_order": DISPLAY_ORDER,
    "software": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
    },
}
write_json(AUDIT_DIR / "statistical_analysis_config.json", CONFIG)
print("No training, model loading, or inference.")
print("Difference direction:", DIFFERENCE_DIRECTION)
print("Bootstrap:", N_BOOTSTRAPS, "paired ImageID-level resamples; seed:", BOOTSTRAP_SEED)


No training, model loading, or inference.
Difference direction: Full Proposed - No-Sampler
Bootstrap: 5000 paired ImageID-level resamples; seed: 42


In [2]:
def discover_prediction_file(root, fold):
    fold_dir = root / f"fold_{fold}"
    candidates = []
    for path in sorted(fold_dir.glob("*.csv")):
        frame = pd.read_csv(path, nrows=3)
        columns = set(frame.columns)
        if "true" in columns and all(f"prob_{name}" in columns for name in CLASS_NAMES):
            candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one exact probability file in {fold_dir}; found {candidates}")
    return candidates[0]

def discover_report_file(root, fold):
    fold_dir = root / f"fold_{fold}"
    reports = sorted(path for path in fold_dir.glob("*.csv") if "report" in path.name.lower())
    return reports[0] if reports else None

def mapping_from_notebook(path):
    source = Path(path).read_text(encoding="utf-8")
    match = re.search(r"label_map\s*=\s*(\{[^}]+\})", source)
    if not match:
        raise RuntimeError(f"Could not verify label mapping in {path}")
    mapping = ast.literal_eval(match.group(1))
    return {str(key): int(value) for key, value in mapping.items()}, source

proposed_mapping, proposed_source = mapping_from_notebook(PROPOSED_NOTEBOOK)
no_sampler_mapping, no_sampler_source = mapping_from_notebook(NO_SAMPLER_NOTEBOOK)
assert proposed_mapping == no_sampler_mapping == LABEL_MAP
assert "WeightedRandomSampler" in proposed_source
assert "WeightedRandomSampler" not in no_sampler_source
assert "test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False" in proposed_source
assert "test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False" in no_sampler_source
print("Verified numeric class mapping:", proposed_mapping)
print("Manuscript display order:", DISPLAY_ORDER)

prediction_paths = {"Full Proposed": {}, "No-Sampler": {}}
roots = {"Full Proposed": PROPOSED_ROOT, "No-Sampler": NO_SAMPLER_ROOT}
print("\nSAVED RESULT INVENTORY")
for model_name, root in roots.items():
    print(f"\n{model_name.upper()}")
    for fold in FOLDS:
        prediction_path = discover_prediction_file(root, fold)
        report_path = discover_report_file(root, fold)
        frame = pd.read_csv(prediction_path)
        prediction_paths[model_name][fold] = prediction_path
        exact = len(frame) == N_IMAGES and "true" in frame.columns and all(f"prob_{name}" in frame.columns for name in CLASS_NAMES)
        print(f"Fold {fold}:")
        print("  prediction file:", prediction_path)
        print("  classification report:", report_path if report_path else "NOT FOUND")
        print("  rows:", len(frame))
        print("  exact paired analysis possible:", "YES" if exact else "NO")
        if not exact:
            raise RuntimeError(
                "Exact paired bootstrap comparison cannot be performed from aggregate "
                "classification reports alone. Per-image fixed-test predictions are required."
            )

test_frame = pd.read_csv(TEST_SPLIT, dtype={"ImageID": str})
assert len(test_frame) == N_IMAGES
assert test_frame.ImageID.nunique() == N_IMAGES
test_ids = test_frame.ImageID.astype(str).to_numpy()
y_true = test_frame.GroupLabel.astype(int).to_numpy()
assert sorted(np.unique(y_true).tolist()) == list(range(6))
test_hash = sha256_file(TEST_SPLIT)
for fold in FOLDS:
    assert sha256_file(PROJECT_ROOT / "folds" / f"fold_{fold}" / "test.csv") == test_hash

probability_columns = [f"prob_{name}" for name in CLASS_NAMES]
arrays = {"Full Proposed": [], "No-Sampler": []}
resolved_rows = {"Full Proposed": [], "No-Sampler": []}
input_hashes = {"Full Proposed": {}, "No-Sampler": {}}
for model_name in roots:
    for fold in FOLDS:
        path = prediction_paths[model_name][fold]
        frame = pd.read_csv(path)
        labels = frame["true"].astype(int).to_numpy()
        probabilities = frame[probability_columns].to_numpy(float)
        assert len(frame) == N_IMAGES
        assert np.array_equal(labels, y_true)
        assert np.isfinite(probabilities).all()
        assert np.allclose(probabilities.sum(axis=1), 1.0, atol=2e-6)
        resolved = pd.DataFrame({
            "Model": model_name, "Fold": fold, "ImageID": test_ids,
            "true_label": [CLASS_NAMES[index] for index in labels],
            "predicted_label": [CLASS_NAMES[index] for index in probabilities.argmax(axis=1)],
        })
        for index, class_name in enumerate(CLASS_NAMES):
            resolved[f"probability {class_name}"] = probabilities[:, index]
        assert resolved.ImageID.nunique() == N_IMAGES
        arrays[model_name].append(probabilities)
        resolved_rows[model_name].append(resolved)
        input_hashes[model_name][f"fold_{fold}"] = sha256_file(path)

proposed_probs = np.stack(arrays["Full Proposed"])
no_sampler_probs = np.stack(arrays["No-Sampler"])
assert proposed_probs.shape == no_sampler_probs.shape == (5, N_IMAGES, 6)
proposed_resolved = pd.concat(resolved_rows["Full Proposed"], ignore_index=True)
no_sampler_resolved = pd.concat(resolved_rows["No-Sampler"], ignore_index=True)
proposed_resolved.to_csv(INPUTS_DIR / "full_proposed_predictions_resolved_imageids.csv", index=False)
no_sampler_resolved.to_csv(INPUTS_DIR / "no_sampler_predictions_resolved_imageids.csv", index=False)

for fold in FOLDS:
    p = proposed_resolved[proposed_resolved.Fold == fold].sort_values("ImageID")
    n = no_sampler_resolved[no_sampler_resolved.Fold == fold].sort_values("ImageID")
    assert np.array_equal(p.ImageID.to_numpy(), n.ImageID.to_numpy())
    assert np.array_equal(p.true_label.to_numpy(), n.true_label.to_numpy())

print("\nTest cohort size:\n719")
print("\nImageID agreement:\nPASS")
print("\nGround-truth agreement:\nPASS")
print("\nFold consistency:\nPASS")
integrity = {
    "valid": True, "fixed_test_size": N_IMAGES, "imageid_agreement": "PASS",
    "ground_truth_agreement": "PASS", "fold_consistency": "PASS",
    "test_split_sha256": test_hash, "class_mapping": LABEL_MAP,
    "manuscript_display_order": DISPLAY_ORDER,
    "probability_input_sha256": input_hashes,
    "imageid_resolution": "Exact fixed test.csv row order; both finalized notebooks save probability rows from test DataLoaders with shuffle=False.",
    "proposed_sampler_verified": True, "no_sampler_verified": True,
}
write_json(AUDIT_DIR / "input_prediction_integrity.json", integrity)


Verified numeric class mapping: {'AD': 0, 'CN': 1, 'EMCI': 2, 'LMCI': 3, 'MCI': 4, 'SMC': 5}
Manuscript display order: ['CN', 'SMC', 'EMCI', 'MCI', 'LMCI', 'AD']

SAVED RESULT INVENTORY

FULL PROPOSED
Fold 1:
  prediction file: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/Results/Proposed_Model/fold_1/probs_fold_1.csv
  classification report: NOT FOUND
  rows: 719
  exact paired analysis possible: YES
Fold 2:
  prediction file: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/Results/Proposed_Model/fold_2/probs_fold_2.csv
  classification report: NOT FOUND
  rows: 719
  exact paired analysis possible: YES
Fold 3:
  prediction file: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimers 6 class classification CBM/Results/Proposed_Model/fold_3/probs_fold_3.csv
  classification report: NOT FOUND
  rows: 719
  exact paired analysis possible: YES
Fold 4:
  prediction file: /media/faizaan/4TB/1_DATA_PROJECTS/Projects/Alheimer

In [3]:
def metric_dict(labels, probabilities):
    predictions = probabilities.argmax(axis=1)
    matrix = np.bincount(labels * 6 + predictions, minlength=36).reshape(6, 6).astype(float)
    support = matrix.sum(axis=1)
    predicted_count = matrix.sum(axis=0)
    true_positive = np.diag(matrix)
    precision = np.divide(true_positive, predicted_count, out=np.zeros(6), where=predicted_count > 0)
    recall = np.divide(true_positive, support, out=np.zeros(6), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(6), where=(precision + recall) > 0)
    total = matrix.sum()
    correct = np.trace(matrix)
    numerator = correct * total - np.dot(predicted_count, support)
    denominator = np.sqrt(
        (total**2 - np.dot(predicted_count, predicted_count))
        * (total**2 - np.dot(support, support))
    )
    mcc = float(numerator / denominator) if denominator else 0.0
    auc_values = []
    for class_index in range(6):
        binary = (labels == class_index).astype(int)
        auc_values.append(
            float(roc_auc_score(binary, probabilities[:, class_index]))
            if np.unique(binary).size == 2 else np.nan
        )
    result = {
        "Accuracy": float(correct / total),
        "Macro Precision": float(precision.mean()),
        "Macro Recall": float(recall.mean()),
        "Macro F1": float(f1.mean()),
        "MCC": mcc,
        "Macro AUC": float(np.nanmean(auc_values)),
        "SMC Precision": float(precision[LABEL_MAP["SMC"]]),
        "SMC Recall": float(recall[LABEL_MAP["SMC"]]),
        "SMC F1": float(f1[LABEL_MAP["SMC"]]),
        "SMC AUC": float(auc_values[LABEL_MAP["SMC"]]),
    }
    result.update({f"{name} F1": float(f1[index]) for index, name in enumerate(CLASS_NAMES)})
    return result

metric_column_names = {
    "Accuracy": "Accuracy", "Macro Precision": "Macro_Precision",
    "Macro Recall": "Macro_Recall", "Macro F1": "Macro_F1",
    "MCC": "MCC", "Macro AUC": "Macro_AUC",
    "SMC Precision": "SMC_Precision", "SMC Recall": "SMC_Recall",
    "SMC F1": "SMC_F1", "SMC AUC": "SMC_AUC",
}
checkpoint_rows = []
checkpoint_metric_objects = {"Full Proposed": {}, "No-Sampler": {}}
for model_name, probabilities in [("Full Proposed", proposed_probs), ("No-Sampler", no_sampler_probs)]:
    for fold in FOLDS:
        metrics = metric_dict(y_true, probabilities[fold - 1])
        checkpoint_metric_objects[model_name][fold] = metrics
        checkpoint_rows.append({
            "Model": model_name, "Fold": fold,
            **{column: metrics[metric] for metric, column in metric_column_names.items()},
        })
checkpoint_metrics = pd.DataFrame(checkpoint_rows)
checkpoint_metrics.to_csv(TABLES_DIR / "checkpoint_metrics_Proposed_vs_NoSampler.csv", index=False)

summary_rows = []
for model_name in ["Full Proposed", "No-Sampler"]:
    frame = checkpoint_metrics[checkpoint_metrics.Model == model_name]
    for statistic, function in [("Mean", lambda s: s.mean()), ("Sample SD", lambda s: s.std(ddof=1))]:
        summary_rows.append({
            "Model": model_name, "Statistic": statistic,
            **{column: float(function(frame[column])) for column in metric_column_names.values()},
        })
checkpoint_summary = pd.DataFrame(summary_rows)
checkpoint_summary.to_csv(TABLES_DIR / "checkpoint_mean_SD_summary.csv", index=False)

observed = {"Full Proposed": {}, "No-Sampler": {}}
observed_sd = {"Full Proposed": {}, "No-Sampler": {}}
for model_name in observed:
    for metric in ALL_METRICS:
        values = [checkpoint_metric_objects[model_name][fold][metric] for fold in FOLDS]
        observed[model_name][metric] = float(np.mean(values))
        observed_sd[model_name][metric] = float(np.std(values, ddof=1))

for model_name, root in roots.items():
    published = pd.read_csv(root / "test_metrics_all_folds.csv").sort_values("fold")
    rows = checkpoint_metrics[checkpoint_metrics.Model == model_name].sort_values("Fold")
    for source, target in {"acc": "Accuracy", "prec": "Macro_Precision", "rec": "Macro_Recall", "f1": "Macro_F1", "auc": "Macro_AUC"}.items():
        assert np.allclose(published[source], rows[target], atol=2e-6)
print(checkpoint_metrics.to_string(index=False))
print("\nFive-checkpoint mean and sample SD")
print(checkpoint_summary.to_string(index=False))


        Model  Fold  Accuracy  Macro_Precision  Macro_Recall  Macro_F1      MCC  Macro_AUC  SMC_Precision  SMC_Recall   SMC_F1  SMC_AUC
Full Proposed     1  0.917942         0.929912      0.889123  0.905574 0.895450   0.950724       1.000000    0.740741 0.851064 0.810158
Full Proposed     2  0.926287         0.927238      0.874779  0.895366 0.905932   0.952428       0.944444    0.629630 0.755556 0.840131
Full Proposed     3  0.821975         0.805646      0.829280  0.810499 0.780298   0.941884       0.571429    0.740741 0.645161 0.859398
Full Proposed     4  0.912378         0.914440      0.877354  0.893707 0.888138   0.947117       0.909091    0.740741 0.816327 0.832638
Full Proposed     5  0.930459         0.940324      0.874904  0.896875 0.911272   0.939356       1.000000    0.555556 0.714286 0.712321
   No-Sampler     1  0.884562         0.892030      0.844273  0.863660 0.852619   0.944946       0.900000    0.666667 0.765957 0.796296
   No-Sampler     2  0.909597         0.879292  

In [4]:
def checkpoint_mean_metrics(probabilities, indices):
    values = [metric_dict(y_true[indices], probabilities[fold - 1, indices]) for fold in FOLDS]
    return {metric: float(np.nanmean([value[metric] for value in values])) for metric in ALL_METRICS}

rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_proposed = {metric: np.empty(N_BOOTSTRAPS) for metric in ALL_METRICS}
bootstrap_no_sampler = {metric: np.empty(N_BOOTSTRAPS) for metric in ALL_METRICS}
for iteration in range(N_BOOTSTRAPS):
    indices = rng.integers(0, N_IMAGES, size=N_IMAGES)
    proposed_value = checkpoint_mean_metrics(proposed_probs, indices)
    no_sampler_value = checkpoint_mean_metrics(no_sampler_probs, indices)
    for metric in ALL_METRICS:
        bootstrap_proposed[metric][iteration] = proposed_value[metric]
        bootstrap_no_sampler[metric][iteration] = no_sampler_value[metric]
    if (iteration + 1) % 500 == 0:
        print(f"Completed {iteration + 1}/{N_BOOTSTRAPS} paired ImageID-level bootstrap resamples")

bootstrap_difference = {
    metric: bootstrap_proposed[metric] - bootstrap_no_sampler[metric]
    for metric in ALL_METRICS
}
pd.DataFrame(bootstrap_proposed).to_csv(BOOTSTRAP_DIR / "proposed_bootstrap_distributions.csv", index=False)
pd.DataFrame(bootstrap_no_sampler).to_csv(BOOTSTRAP_DIR / "no_sampler_bootstrap_distributions.csv", index=False)
pd.DataFrame(bootstrap_difference).to_csv(BOOTSTRAP_DIR / "paired_difference_bootstrap_distributions.csv", index=False)

def percentile_ci(values):
    return tuple(float(value) for value in np.nanpercentile(values, [2.5, 97.5]))

def model_ci_frame(model_name, observed_values, observed_sds, distributions):
    rows = []
    for metric in REPORT_METRICS:
        low, high = percentile_ci(distributions[metric])
        rows.append({
            "Model": model_name, "Metric": metric,
            "Observed_Mean": observed_values[metric],
            "Observed_Sample_SD": observed_sds[metric],
            "CI_Low": low, "CI_High": high,
            "CI_Method": "95% paired ImageID-level percentile bootstrap",
        })
    return pd.DataFrame(rows)

proposed_ci = model_ci_frame("Full Proposed", observed["Full Proposed"], observed_sd["Full Proposed"], bootstrap_proposed)
no_sampler_ci = model_ci_frame("No-Sampler", observed["No-Sampler"], observed_sd["No-Sampler"], bootstrap_no_sampler)
proposed_ci.to_csv(BOOTSTRAP_DIR / "proposed_metric_CI.csv", index=False)
no_sampler_ci.to_csv(BOOTSTRAP_DIR / "no_sampler_metric_CI.csv", index=False)

difference_rows = []
for metric in REPORT_METRICS:
    low, high = percentile_ci(bootstrap_difference[metric])
    difference_rows.append({
        "Metric": metric,
        "Proposed_Observed_Mean": observed["Full Proposed"][metric],
        "No_Sampler_Observed_Mean": observed["No-Sampler"][metric],
        "Difference": observed["Full Proposed"][metric] - observed["No-Sampler"][metric],
        "CI_Low": low, "CI_High": high,
        "CI_Excludes_Zero": bool(low > 0 or high < 0),
        "Interpretation": "paired 95% CI excludes zero" if (low > 0 or high < 0) else "paired 95% CI includes zero",
    })
paired_difference = pd.DataFrame(difference_rows)
paired_difference.to_csv(BOOTSTRAP_DIR / "paired_difference_CI.csv", index=False)

classwise_rows = []
for class_name in DISPLAY_ORDER:
    metric = f"{class_name} F1"
    p_low, p_high = percentile_ci(bootstrap_proposed[metric])
    n_low, n_high = percentile_ci(bootstrap_no_sampler[metric])
    d_low, d_high = percentile_ci(bootstrap_difference[metric])
    classwise_rows.append({
        "Class": class_name,
        "Full_Proposed_Mean": observed["Full Proposed"][metric],
        "Full_Proposed_Sample_SD": observed_sd["Full Proposed"][metric],
        "Full_Proposed_CI_Low": p_low, "Full_Proposed_CI_High": p_high,
        "No_Sampler_Mean": observed["No-Sampler"][metric],
        "No_Sampler_Sample_SD": observed_sd["No-Sampler"][metric],
        "No_Sampler_CI_Low": n_low, "No_Sampler_CI_High": n_high,
        "Difference_Full_Proposed_minus_No_Sampler": observed["Full Proposed"][metric] - observed["No-Sampler"][metric],
        "Difference_CI_Low": d_low, "Difference_CI_High": d_high,
        "CI_Excludes_Zero": bool(d_low > 0 or d_high < 0),
    })
classwise_comparison = pd.DataFrame(classwise_rows)
classwise_comparison.to_csv(BOOTSTRAP_DIR / "classwise_F1_comparison.csv", index=False)
print("\nPaired differences")
print(paired_difference.to_string(index=False))


Completed 500/5000 paired ImageID-level bootstrap resamples


Completed 1000/5000 paired ImageID-level bootstrap resamples


Completed 1500/5000 paired ImageID-level bootstrap resamples


Completed 2000/5000 paired ImageID-level bootstrap resamples


Completed 2500/5000 paired ImageID-level bootstrap resamples


Completed 3000/5000 paired ImageID-level bootstrap resamples


Completed 3500/5000 paired ImageID-level bootstrap resamples


Completed 4000/5000 paired ImageID-level bootstrap resamples


Completed 4500/5000 paired ImageID-level bootstrap resamples


Completed 5000/5000 paired ImageID-level bootstrap resamples



Paired differences
         Metric  Proposed_Observed_Mean  No_Sampler_Observed_Mean  Difference    CI_Low  CI_High  CI_Excludes_Zero              Interpretation
       Accuracy                0.901808                  0.910431   -0.008623 -0.018081 0.001113             False paired 95% CI includes zero
Macro Precision                0.903512                  0.900605    0.002907 -0.010485 0.017383             False paired 95% CI includes zero
   Macro Recall                0.869088                  0.866215    0.002873 -0.006182 0.012253             False paired 95% CI includes zero
       Macro F1                0.880404                  0.880471   -0.000067 -0.010213 0.010753             False paired 95% CI includes zero
            MCC                0.876218                  0.885696   -0.009478 -0.020959 0.002529             False paired 95% CI includes zero
      Macro AUC                0.946302                  0.950703   -0.004401 -0.009669 0.000972             False paired 

In [5]:
main_metrics = ["Accuracy", "Macro F1", "MCC", "Macro AUC", "SMC F1", "SMC Recall"]
manuscript_rows = []
for metric in main_metrics:
    p_row = proposed_ci[proposed_ci.Metric == metric].iloc[0]
    n_row = no_sampler_ci[no_sampler_ci.Metric == metric].iloc[0]
    d_row = paired_difference[paired_difference.Metric == metric].iloc[0]
    manuscript_rows.append({
        "Metric": metric,
        "Full Proposed mean ± SD": f"{p_row.Observed_Mean:.6f} ± {p_row.Observed_Sample_SD:.6f}",
        "Full Proposed 95% CI": f"{p_row.CI_Low:.6f} to {p_row.CI_High:.6f}",
        "No-Sampler mean ± SD": f"{n_row.Observed_Mean:.6f} ± {n_row.Observed_Sample_SD:.6f}",
        "No-Sampler 95% CI": f"{n_row.CI_Low:.6f} to {n_row.CI_High:.6f}",
        "Difference Full Proposed - No-Sampler": d_row.Difference,
        "Difference 95% CI": f"{d_row.CI_Low:.6f} to {d_row.CI_High:.6f}",
        "CI excludes zero YES/NO": "YES" if d_row.CI_Excludes_Zero else "NO",
    })
manuscript_table = pd.DataFrame(manuscript_rows)
manuscript_table.to_csv(TABLES_DIR / "manuscript_ablation_statistical_summary.csv", index=False)

forest = paired_difference.set_index("Metric").loc[main_metrics].reset_index()
y = np.arange(len(forest))
fig, axis = plt.subplots(figsize=(8.2, 5.6))
axis.errorbar(
    forest.Difference, y,
    xerr=[forest.Difference - forest.CI_Low, forest.CI_High - forest.Difference],
    fmt="o", color="#4C78A8", ecolor="#4C78A8", capsize=4,
)
axis.axvline(0, color="black", linestyle="--", linewidth=1)
axis.set_yticks(y, forest.Metric)
axis.invert_yaxis()
axis.set_xlabel("Full Proposed - No-Sampler")
axis.set_title("Paired bootstrap differences with 95% CIs")
axis.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "paired_difference_CI_forest_plot.png", dpi=PLOT_DPI, bbox_inches="tight")
fig.savefig(FIGURES_DIR / "paired_difference_CI_forest_plot.pdf", bbox_inches="tight")
plt.close(fig)

smc_metric = "SMC F1"
smc_distribution = bootstrap_difference[smc_metric]
smc_row = paired_difference[paired_difference.Metric == smc_metric].iloc[0]
fig, axis = plt.subplots(figsize=(8.0, 5.4))
axis.hist(smc_distribution, bins=55, color="#6C8EAD", alpha=0.88, edgecolor="white", linewidth=0.3)
axis.axvline(0, color="black", linestyle="--", label="No difference")
axis.axvline(smc_row.Difference, color="#D9544D", linewidth=2, label=f"Observed={smc_row.Difference:.4f}")
axis.axvspan(smc_row.CI_Low, smc_row.CI_High, color="#F4A261", alpha=0.20, label=f"95% CI [{smc_row.CI_Low:.4f}, {smc_row.CI_High:.4f}]")
axis.set_xlabel("Full Proposed - No-Sampler SMC F1")
axis.set_ylabel("Bootstrap count")
axis.set_title("Paired ImageID-level bootstrap: SMC F1 difference")
axis.legend()
axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "SMC_F1_paired_bootstrap_distribution.png", dpi=PLOT_DPI, bbox_inches="tight")
plt.close(fig)

required = [
    TABLES_DIR / "checkpoint_metrics_Proposed_vs_NoSampler.csv",
    TABLES_DIR / "checkpoint_mean_SD_summary.csv",
    TABLES_DIR / "manuscript_ablation_statistical_summary.csv",
    BOOTSTRAP_DIR / "proposed_metric_CI.csv",
    BOOTSTRAP_DIR / "no_sampler_metric_CI.csv",
    BOOTSTRAP_DIR / "paired_difference_CI.csv",
    BOOTSTRAP_DIR / "classwise_F1_comparison.csv",
    BOOTSTRAP_DIR / "proposed_bootstrap_distributions.csv",
    BOOTSTRAP_DIR / "no_sampler_bootstrap_distributions.csv",
    BOOTSTRAP_DIR / "paired_difference_bootstrap_distributions.csv",
    FIGURES_DIR / "paired_difference_CI_forest_plot.png",
    FIGURES_DIR / "paired_difference_CI_forest_plot.pdf",
    FIGURES_DIR / "SMC_F1_paired_bootstrap_distribution.png",
    AUDIT_DIR / "statistical_analysis_config.json",
    AUDIT_DIR / "input_prediction_integrity.json",
]
assert all(path.is_file() and path.stat().st_size > 0 for path in required)
completion = {
    "valid": True, "completed_utc": datetime.now(timezone.utc).isoformat(),
    "required_outputs_present": len(required), "training_performed": False,
    "inference_performed": False, "existing_files_modified": False,
    "output_sha256": {str(path.relative_to(OUTPUT_ROOT)): sha256_file(path) for path in required},
}
write_json(AUDIT_DIR / "analysis_completion.json", completion)

print("==================================================")
print("PROPOSED VS NO-SAMPLER STATISTICAL ANALYSIS")
print("==================================================")
print("\nFixed test cohort:\n719 MRI volumes")
print("\nFull Proposed checkpoints:\n5")
print("\nNo-Sampler checkpoints:\n5")
print("\nBootstrap:\n5,000 paired resamples")
print("\nConfidence interval:\n95% percentile bootstrap")
print("\nDifference:\nFull Proposed - No-Sampler")
print("\nPRIMARY METRICS")
for metric in ["Accuracy", "Macro F1", "MCC", "Macro AUC"]:
    p = proposed_ci[proposed_ci.Metric == metric].iloc[0]
    n = no_sampler_ci[no_sampler_ci.Metric == metric].iloc[0]
    d = paired_difference[paired_difference.Metric == metric].iloc[0]
    print(f"\n{metric}:")
    print("Proposed:", f"{p.Observed_Mean:.6f} ± {p.Observed_Sample_SD:.6f}")
    print("No-Sampler:", f"{n.Observed_Mean:.6f} ± {n.Observed_Sample_SD:.6f}")
    print("Difference:", f"{d.Difference:.6f}")
    print("95% CI:", f"{d.CI_Low:.6f} to {d.CI_High:.6f}")
    print("CI excludes zero:", "YES" if d.CI_Excludes_Zero else "NO")
print("\nMINORITY-CLASS METRICS")
for metric in ["SMC F1", "SMC Recall"]:
    p = proposed_ci[proposed_ci.Metric == metric].iloc[0]
    n = no_sampler_ci[no_sampler_ci.Metric == metric].iloc[0]
    d = paired_difference[paired_difference.Metric == metric].iloc[0]
    print(f"\n{metric}:")
    print("Proposed:", f"{p.Observed_Mean:.6f} ± {p.Observed_Sample_SD:.6f}")
    print("No-Sampler:", f"{n.Observed_Mean:.6f} ± {n.Observed_Sample_SD:.6f}")
    print("Difference:", f"{d.Difference:.6f}")
    print("95% CI:", f"{d.CI_Low:.6f} to {d.CI_High:.6f}")
    print("CI excludes zero:", "YES" if d.CI_Excludes_Zero else "NO")
print("\nTraining rerun:\nNO")
print("\nInference rerun:\nNO")
print("\nEnsemble analysis:\nNO")
print("\nExisting files modified:\nNONE")
print("\n==================================================")
print("New notebook:", NOTEBOOK_PATH.resolve())
print("New output files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path.resolve())


PROPOSED VS NO-SAMPLER STATISTICAL ANALYSIS

Fixed test cohort:
719 MRI volumes

Full Proposed checkpoints:
5

No-Sampler checkpoints:
5

Bootstrap:
5,000 paired resamples

Confidence interval:
95% percentile bootstrap

Difference:
Full Proposed - No-Sampler

PRIMARY METRICS

Accuracy:
Proposed: 0.901808 ± 0.045181
No-Sampler: 0.910431 ± 0.016562
Difference: -0.008623
95% CI: -0.018081 to 0.001113
CI excludes zero: NO

Macro F1:
Proposed: 0.880404 ± 0.039346
No-Sampler: 0.880471 ± 0.026582
Difference: -0.000067
95% CI: -0.010213 to 0.010753
CI excludes zero: NO

MCC:
Proposed: 0.876218 ± 0.054370
No-Sampler: 0.885696 ± 0.021134
Difference: -0.009478
95% CI: -0.020959 to 0.002529
CI excludes zero: NO

Macro AUC:
Proposed: 0.946302 ± 0.005602
No-Sampler: 0.950703 ± 0.005305
Difference: -0.004401
95% CI: -0.009669 to 0.000972
CI excludes zero: NO

MINORITY-CLASS METRICS

SMC F1:
Proposed: 0.756479 ± 0.081702
No-Sampler: 0.720982 ± 0.131447
Difference: 0.035497
95% CI: 0.002207 to 0.073563